<a href="https://colab.research.google.com/github/sadiq937/Shaik_DM/blob/main/PS4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [90]:
!pip install fuzzywuzzy

In [ ]:
!pip install python-Levenshtein

In [ ]:
!pip install yfinance

In [112]:
import requests
import os
import pandas as pd
import re
import time
import plotly.express as px
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import folium
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt


In [113]:
# FRED API Key (Replace with your own API key)
FRED_API_KEY = "724bdeda0e13567e197e4ec09e79e655"

# Function to fetch data from FRED
def fetch_fred_data(series_id, start_date="2000-01-01", end_date=str(datetime.today().date())):
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json&observation_start={start_date}&observation_end={end_date}"
    response = requests.get(url).json()
    data = response.get("observations", [])
    return pd.DataFrame([{"date": obs["date"], series_id: float(obs["value"])} for obs in data if obs["value"] != "."])

In [126]:
# S&P 500 Index
sp500_df = fetch_fred_data("SP500")
sp500_df.head()

,date,SP500
0,2015-03-25,2061.05
1,2015-03-26,2056.15
2,2015-03-27,2061.02
3,2015-03-30,2086.24
4,2015-03-31,2067.89


In [127]:
# Consumer Price Index
inflation_df = fetch_fred_data("CPIAUCSL")
inflation_df.head()

,date,CPIAUCSL
0,2000-01-01,169.3
1,2000-02-01,170.0
2,2000-03-01,171.0
3,2000-04-01,170.9
4,2000-05-01,171.2


In [128]:
# Fedral Funds Rate
interest_rate_df = fetch_fred_data("FEDFUNDS")
interest_rate_df.head()

,date,FEDFUNDS
0,2000-01-01,5.45
1,2000-02-01,5.73
2,2000-03-01,5.85
3,2000-04-01,6.02
4,2000-05-01,6.27


In [129]:
# Real GDP Growth Rate
gdp_df = fetch_fred_data("A191RL1Q225SBEA")
gdp_df.head()

,date,A191RL1Q225SBEA
0,2000-01-01,1.5
1,2000-04-01,7.5
2,2000-07-01,0.4
3,2000-10-01,2.4
4,2001-01-01,-1.3


In [130]:
# Unemployment Rate
unemployment_df = fetch_fred_data("UNRATE")
unemployment_df.head()

,date,UNRATE
0,2000-01-01,4.0
1,2000-02-01,4.1
2,2000-03-01,4.0
3,2000-04-01,3.8
4,2000-05-01,4.0


In [131]:
# Merging data into a single DataFrame
macro_df = sp500_df.merge(inflation_df, on="date", how="inner")\
                      .merge(interest_rate_df, on="date", how="inner")\
                      .merge(gdp_df, on="date", how="inner")\
                      .merge(unemployment_df, on="date", how="inner")

macro_df["date"] = pd.to_datetime(macro_df["date"])
macro_df.set_index("date", inplace=True)
macro_df.head()

,SP500,CPIAUCSL,FEDFUNDS,A191RL1Q225SBEA,UNRATE
date,,,,,
2015-04-01,2059.69,236.222,0.12,2.5,5.4
2015-07-01,2077.42,238.034,0.13,1.6,5.2
2015-10-01,1923.82,237.733,0.12,0.7,5.0
2016-04-01,2072.78,238.992,0.37,1.3,5.1
2016-07-01,2102.95,240.101,0.39,2.9,4.8


In [132]:
# Compute percentage changes
pct_change_df = macro_df.pct_change().dropna()
pct_change_df.head()

,SP500,CPIAUCSL,FEDFUNDS,A191RL1Q225SBEA,UNRATE
date,,,,,
2015-07-01,0.008608,0.007671,0.083333,-0.360000,-0.037037
2015-10-01,-0.073938,-0.001265,-0.076923,-0.562500,-0.038462
2016-04-01,0.077429,0.005296,2.083333,0.857143,0.020000
2016-07-01,0.014555,0.004640,0.054054,1.230769,-0.058824
2018-10-01,0.390708,0.052774,4.615385,-0.793103,-0.208333


In [133]:
# Stack/Unstack transformation
long_format_df = pct_change_df.stack().reset_index()
long_format_df.columns = ["date", "indicator", "value"]
long_format_df.head()

,date,indicator,value
0,2015-07-01,SP500,0.008608
1,2015-07-01,CPIAUCSL,0.007671
2,2015-07-01,FEDFUNDS,0.083333
3,2015-07-01,A191RL1Q225SBEA,-0.360000
4,2015-07-01,UNRATE,-0.037037


In [134]:
# Categorizing market trends
def categorize_market(value):
    if value > 0.02:
        return "Boom"
    elif value < -0.02:
        return "Downturn"
    else:
        return "Stable"

pct_change_df["Market_Trend"] = pct_change_df["SP500"].apply(categorize_market)
pct_change_df.head()

,SP500,CPIAUCSL,FEDFUNDS,A191RL1Q225SBEA,UNRATE,Market_Trend
date,,,,,,
2015-07-01,0.008608,0.007671,0.083333,-0.360000,-0.037037,Stable
2015-10-01,-0.073938,-0.001265,-0.076923,-0.562500,-0.038462,Downturn
2016-04-01,0.077429,0.005296,2.083333,0.857143,0.020000,Boom
2016-07-01,0.014555,0.004640,0.054054,1.230769,-0.058824,Stable
2018-10-01,0.390708,0.052774,4.615385,-0.793103,-0.208333,Boom


In [135]:
# Fuzzy Matching Example
example_df = pd.DataFrame({"Indicator_Names": ["CPIAUCSL", "FED Funds Rate", "SP 500", "CPI"]})
example_df["Matched_Names"] = example_df["Indicator_Names"].apply(lambda x: "Consumer Price Index" if fuzz.ratio(x, "Consumer Price Index") > 70 else x)

In [136]:
# Reshape (Pivot) Example
pivot_df = long_format_df.pivot(index="date", columns="indicator", values="value")

# Descriptive statistics
numerical_df = pct_change_df.select_dtypes(include=['number'])
correlation_matrix_df = numerical_df.corr()

# Visualization using Plotly
fig1 = px.imshow(correlation_matrix_df, text_auto=True, color_continuous_scale='RdBu_r', title="Correlation Between Macroeconomic Indicators")
fig1.show()

# Line plot for S&P 500 trends
df_reset = pct_change_df.reset_index()
fig2 = px.line(df_reset, x="date", y="SP500", title="S&P 500 Trend Over Time", labels={"SP500": "S&P 500 Returns"})
fig2.show()

# Scatter plot of Inflation vs. Interest Rates
fig3 = px.scatter(df_reset, x="CPIAUCSL", y="FEDFUNDS", title="Inflation vs. Interest Rates", labels={"CPIAUCSL": "Inflation (CPI)", "FEDFUNDS": "Interest Rate"}, trendline="ols")
fig3.show()

# Bar chart for market trend categories
market_trend_counts_df = pct_change_df["Market_Trend"].value_counts().reset_index()
market_trend_counts_df.columns = ["Trend", "Count"]
fig4 = px.bar(market_trend_counts_df, x="Trend", y="Count", title="Market Trend Distribution", color="Trend")
fig4.show()

# Time-series area plot
fig5 = px.area(df_reset, x="date", y=["SP500", "CPIAUCSL", "FEDFUNDS", "A191RL1Q225SBEA", "UNRATE"], title="Macroeconomic Indicators Over Time", labels={"value": "Percentage Change", "variable": "Indicator"})
fig5.show()